<a href="https://colab.research.google.com/github/RAseng77/AIFFEL_QUEST_RS/blob/master/MainQuest/Quest01/MainQuest01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentencepiece

In [ ]:

!mkdir -p ~/work/transformer_chatbot/data/


%cd ~/work/transformer_chatbot/data/


!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv


!ls -l

/home/jovyan/work/transformer_chatbot/data
--2026-02-09 06:03:08--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-02-09 06:03:08--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.4’

ChatbotData.csv.4   100%[===================>] 868.99K  5.35MB/s    in 0.2s    

2026-02-09 06:03:09 (5.35 MB/s) - ‘ChatbotData.csv.4’ saved [889842/88984

In [ ]:
import pandas as pd

data = pd.read_csv('ChatbotData.csv')

print(f"데이터 개수 {len(data)}")
print(data.head())

데이터 개수 11823
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


In [ ]:
MAX_SAMPLES = 50000

In [ ]:
print("전처리 전:", data['Q'][0])
print("전처리 후:", preprocess_sentence(data['Q'][0]))

전처리 전: 12시 땡!
전처리 후: 12시 땡 !


In [ ]:
corpus = []
for sentence in data['Q']:
    corpus.append(preprocess_sentence(sentence))
for sentence in data['A']:
    corpus.append(preprocess_sentence(sentence))

# text 파일로 저장
with open('chatbot_corpus.txt', 'w', encoding='utf-8') as f:
    for sentence in corpus:
        f.write(sentence + '\n')

print("코퍼스 저장 완료: chatbot_corpus.txt")

코퍼스 저장 완료: chatbot_corpus.txt


In [ ]:
import sentencepiece as spm

# 파라미터 설정
input_file = 'chatbot_corpus.txt'
vocab_size = 8000
model_name = 'korean_spm'
model_type = 'unigram'
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

input_argument = '--input={} --model_prefix={} --vocab_size={} --user_defined_symbols={} --model_type={} --pad_id={} --bos_id={} --eos_id={} --unk_id={}'.format(
    input_file, model_name, vocab_size, 'some,symbols', model_type, pad_id, bos_id, eos_id, unk_id
)

spm.SentencePieceTrainer.Train(input_argument)

print("SentencePiece 학습 완료!")

SentencePiece 학습 완료!


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=chatbot_corpus.txt --model_prefix=korean_spm --vocab_size=8000 --user_defined_symbols=some,symbols --model_type=unigram --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot_corpus.txt
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: some
  user_defined_symbols: symbols
  required_chars: 
  byte_fallback: 0
  vocabulary_output_pi

In [ ]:
# 토크나이저 로드
sp = spm.SentencePieceProcessor()
sp.Load(f'{model_name}.model')

True

In [ ]:
MAX_SAMPLES = 50000
MAX_LENGTH = 40

In [ ]:
dataset = GPTChatbotDataset(data, sp, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import re


# 전처리 및 데이터셋 정의
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    return sentence.strip()

class GPTChatbotDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.questions = df['Q'].tolist()
        self.answers = df['A'].tolist()
        # GPT-1 논문의 구분자($) 역할을 하는 토큰 ID
        self.delim_id = tokenizer.bos_id()

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q = self.tokenizer.EncodeAsIds(preprocess_sentence(self.questions[idx]))
        a = self.tokenizer.EncodeAsIds(preprocess_sentence(self.answers[idx]))

        # [BOS] + Q + [DELIM] + A + [EOS] 시퀀스 구성
        input_ids = [self.tokenizer.bos_id()] + q + [self.delim_id] + a + [self.tokenizer.eos_id()]

        if len(input_ids) < self.max_len:
            input_ids += [self.tokenizer.pad_id()] * (self.max_len - len(input_ids))
        else:
            input_ids = input_ids[:self.max_len]
        return torch.tensor(input_ids)

# 모델 구성 요소
def gelu(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.depth = d_model // num_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
    def split_heads(self, x, batch_size):
        return x.view(batch_size, -1, self.num_heads, self.depth).permute(0, 2, 1, 3)
    def forward(self, v, k, q, mask):
        batch_size = q.size(0)
        q, k, v = self.split_heads(self.wq(q), batch_size), self.split_heads(self.wk(k), batch_size), self.split_heads(self.wv(v), batch_size)
        scaled_attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.depth)
        if mask is not None: scaled_attention += (mask * -1e9)
        output = torch.matmul(F.softmax(scaled_attention, dim=-1), v)
        return self.dense(output.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.num_heads * self.depth)), None

class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, dff), nn.Lambda(gelu) if hasattr(nn, 'Lambda') else nn.GELU(), nn.Linear(dff, d_model))
        self.layernorm1, self.layernorm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.dropout1, self.dropout2 = nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, mask):
        x = self.layernorm1(x + self.dropout1(self.mha(x, x, x, mask)[0]))
        x = self.layernorm2(x + self.dropout2(self.ffn(x)))
        return x


class GPT1(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size, max_len, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([GPTBlock(d_model, num_heads, dff, dropout) for _ in range(num_layers)])
        self.lm_head = nn.Linear(d_model, vocab_size)
    def forward(self, x):
        mask = torch.max(torch.eq(x, 0).float()[:, None, None, :], torch.triu(torch.ones(x.size(1), x.size(1)), 1).to(x.device))
        x = self.pos_encoding(self.embedding(x) * math.sqrt(x.size(-1)))
        for block in self.blocks: x = block(x, mask)
        return self.lm_head(x)


def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, targets = batch[:, :-1], batch[:, 1:]
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def generate_response(model, tokenizer, question, max_len=40, device='cpu'):
    model.eval()
    # 질문 인코딩 + 시작 토큰 + 구분자
    input_ids = [tokenizer.bos_id()] + tokenizer.EncodeAsIds(preprocess_sentence(question)) + [tokenizer.bos_id()]
    input_tensor = torch.tensor([input_ids]).to(device)

    for _ in range(max_len):
        outputs = model(input_tensor)
        next_token = outputs[:, -1, :].argmax(dim=-1).unsqueeze(0)
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        if next_token.item() == tokenizer.eos_id(): break


    response_ids = input_tensor.squeeze().tolist()[len(input_ids):]
    return tokenizer.DecodeIds(response_ids)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = 8000
max_len = 40

# 모델 초기화
model = GPT1(num_layers=4, d_model=256, num_heads=8, dff=1024, vocab_size=vocab_size, max_len=max_len).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=6.25e-5) # 논문 권장 학습률
criterion = nn.CrossEntropyLoss(ignore_index=0) # 패딩은 무시

epochs = 100

print("학습시작")

for epoch in range(epochs):

    avg_loss = train(model, dataloader, optimizer, criterion, device)

    # 1 에포크마다 현재 상태 출력
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

print("\n학습 완료! 테스트를 진행")
print("-" * 30)

# 테스트 실행
test_question = "오늘 날씨 어때?"
answer = generate_response(model, sp, test_question, device=device)
print(f"질문: {test_question}")
print(f"답변: {answer}")

학습시작
Epoch [1/100] - Loss: 7.0279
Epoch [2/100] - Loss: 5.9836
Epoch [3/100] - Loss: 5.6916
Epoch [4/100] - Loss: 5.4721
Epoch [5/100] - Loss: 5.2874
Epoch [6/100] - Loss: 5.1262
Epoch [7/100] - Loss: 4.9813
Epoch [8/100] - Loss: 4.8518
Epoch [9/100] - Loss: 4.7305
Epoch [10/100] - Loss: 4.6183
Epoch [11/100] - Loss: 4.5142
Epoch [12/100] - Loss: 4.4126
Epoch [13/100] - Loss: 4.3181
Epoch [14/100] - Loss: 4.2250
Epoch [15/100] - Loss: 4.1369
Epoch [16/100] - Loss: 4.0519
Epoch [17/100] - Loss: 3.9694
Epoch [18/100] - Loss: 3.8881
Epoch [19/100] - Loss: 3.8102
Epoch [20/100] - Loss: 3.7341
Epoch [21/100] - Loss: 3.6572
Epoch [22/100] - Loss: 3.5847
Epoch [23/100] - Loss: 3.5131
Epoch [24/100] - Loss: 3.4421
Epoch [25/100] - Loss: 3.3738
Epoch [26/100] - Loss: 3.3051
Epoch [27/100] - Loss: 3.2369
Epoch [28/100] - Loss: 3.1722
Epoch [29/100] - Loss: 3.1081
Epoch [30/100] - Loss: 3.0478
Epoch [31/100] - Loss: 2.9833
Epoch [32/100] - Loss: 2.9227
Epoch [33/100] - Loss: 2.8608
Epoch [34/100]